In [ ]:
import scanpy as sc
import pandas as pd
import squidpy as sq
import numpy as np
from scipy.spatial import *
from sklearn.preprocessing import *

from sklearn.metrics import *
from scipy.spatial.distance import *

# Define the function to compute all metrics for each iteration and save to CSV
def eval(adata, gt_key, spatial_key='spatial', output_csv='evaluation_results.csv'):
    # Initialize lists to store results for each iteration
    metrics_results = {
        'Iteration': [],
        'ARI': [],
        'NMI': [],
        'AMI': [],
        'HOM': [],
        'COM': [],
        'CHAOS': [],
        'PAS': [],
        'MoranI': [],
        'GearyC': []
    }

    # Loop through all the predictions (10 results)
    for i in range(10):
        pred_key = f'refined_pred_{i}'

        # If gt_key is not None, compute the metrics involving the ground truth (ARI, NMI, AMI, HOM, COM)
        if gt_key is not None:
            # Compute ARI
            ARI = compute_ARI(adata, gt_key, pred_key)
            print(ARI)
            metrics_results['Iteration'].append(f'pred_{i}')
            metrics_results['ARI'].append(ARI)

            # Compute NMI
            NMI = compute_NMI(adata, gt_key, pred_key)
            metrics_results['NMI'].append(NMI)

            # Compute AMI
            AMI = compute_AMI(adata, gt_key, pred_key)
            metrics_results['AMI'].append(AMI)

            # Compute HOM
            HOM = compute_HOM(adata, gt_key, pred_key)
            metrics_results['HOM'].append(HOM)

            # Compute COM
            COM = compute_COM(adata, gt_key, pred_key)
            metrics_results['COM'].append(COM)
        else:
            # If gt_key is None, just append NaN or some placeholder value
            metrics_results['Iteration'].append(f'pred_{i}')
            metrics_results['ARI'].append(np.nan)
            metrics_results['NMI'].append(np.nan)
            metrics_results['AMI'].append(np.nan)
            metrics_results['HOM'].append(np.nan)
            metrics_results['COM'].append(np.nan)

        # Compute CHAOS
        CHAOS = compute_CHAOS(adata, pred_key, spatial_key)
        metrics_results['CHAOS'].append(CHAOS)

        # Compute PAS
        PAS = compute_PAS(adata, pred_key, spatial_key)
        metrics_results['PAS'].append(PAS)

        # Compute marker scores (Moran's I and Geary's C)
        moranI, gearyC = marker_score(adata, pred_key)
        metrics_results['MoranI'].append(moranI)
        metrics_results['GearyC'].append(gearyC)

    # Convert results to a DataFrame
    df_results = pd.DataFrame(metrics_results)

    # Calculate the mean and standard deviation for each metric
    mean_std_results = {
        'Metric': ['ARI', 'NMI', 'AMI', 'HOM', 'COM', 'CHAOS', 'PAS', 'MoranI', 'GearyC'],
        'Mean': [
            df_results['ARI'].mean(), df_results['NMI'].mean(), df_results['AMI'].mean(),
            df_results['HOM'].mean(), df_results['COM'].mean(), df_results['CHAOS'].mean(),
            df_results['PAS'].mean(), df_results['MoranI'].mean(), df_results['GearyC'].mean()
        ],
        'Std': [
            df_results['ARI'].std(), df_results['NMI'].std(), df_results['AMI'].std(),
            df_results['HOM'].std(), df_results['COM'].std(), df_results['CHAOS'].std(),
            df_results['PAS'].std(), df_results['MoranI'].std(), df_results['GearyC'].std()
        ]
    }
    
    # Convert mean and std to DataFrame and append to the results
    df_mean_std = pd.DataFrame(mean_std_results)
    df_results = pd.concat([df_results, df_mean_std], ignore_index=True)
    # print(df_results)

    # Save the DataFrame to a CSV file
    df_results.to_csv(output_csv, index=False)
    print(f"Results saved to {output_csv}")

    return df_results


def fx_1NN(i,location_in):
        location_in = np.array(location_in)
        dist_array = distance_matrix(location_in[i,:][None,:],location_in)[0,:]
        dist_array[i] = np.inf
        return np.min(dist_array)


def _compute_CHAOS(clusterlabel, location):

    clusterlabel = np.array(clusterlabel)
    location = np.array(location)
    matched_location = StandardScaler().fit_transform(location)

    clusterlabel_unique = np.unique(clusterlabel)
    dist_val = np.zeros(len(clusterlabel_unique))
    count = 0
    for k in clusterlabel_unique:
        location_cluster = matched_location[clusterlabel==k,:]
        if len(location_cluster)<=2:
            continue
        n_location_cluster = len(location_cluster)
        results = [fx_1NN(i,location_cluster) for i in range(n_location_cluster)]
        dist_val[count] = np.sum(results)
        count = count + 1

    return np.sum(dist_val)/len(clusterlabel)


def fx_kNN(i,location_in,k,cluster_in):

    location_in = np.array(location_in)
    cluster_in = np.array(cluster_in)


    dist_array = distance_matrix(location_in[i,:][None,:],location_in)[0,:]
    dist_array[i] = np.inf
    ind = np.argsort(dist_array)[:k]
    cluster_use = np.array(cluster_in)
    if np.sum(cluster_use[ind]!=cluster_in[i])>(k/2):
        return 1
    else:
        return 0
    

def _compute_PAS(clusterlabel,location):
    
    clusterlabel = np.array(clusterlabel)
    location = np.array(location)
    matched_location = location
    results = [fx_kNN(i,matched_location,k=10,cluster_in=clusterlabel) for i in range(matched_location.shape[0])]
    return np.sum(results)/len(clusterlabel)
    

# Define the individual metrics functions (as previously given)
def compute_ARI(adata, gt_key, pred_key):
    return adjusted_rand_score(adata.obs[gt_key], adata.obs[pred_key])

def compute_NMI(adata, gt_key, pred_key):
    return normalized_mutual_info_score(adata.obs[gt_key], adata.obs[pred_key])

def compute_AMI(adata, gt_key, pred_key):
    return adjusted_mutual_info_score(adata.obs[gt_key], adata.obs[pred_key])

def compute_HOM(adata, gt_key, pred_key):
    return homogeneity_score(adata.obs[gt_key], adata.obs[pred_key])

def compute_COM(adata, gt_key, pred_key):
    return completeness_score(adata.obs[gt_key], adata.obs[pred_key])

def compute_CHAOS(adata, pred_key, spatial_key='spatial'):
    return _compute_CHAOS(adata.obs[pred_key], adata.obsm[spatial_key])

def compute_PAS(adata, pred_key, spatial_key='spatial'):
    return _compute_PAS(adata.obs[pred_key], adata.obsm[spatial_key])

def marker_score(adata, domain_key, top_n=5):
    adata = adata.copy()

    # Ensure that the domain_key is a categorical type
    adata.obs[domain_key] = adata.obs[domain_key].astype('category')

    count_dict = adata.obs[domain_key].value_counts()
    adata = adata[adata.obs[domain_key].isin(count_dict.keys()[count_dict>3].values)]
    sc.pp.normalize_per_cell(adata)
    sc.pp.log1p(adata)
    sc.tl.rank_genes_groups(adata, groupby=domain_key)
    selected_genes = []
    for i in range(top_n):
        toadd = list(adata.uns['rank_genes_groups']['names'][i])
        selected_genes.extend(toadd)
    selected_genes = np.unique(selected_genes)
    sq.gr.spatial_neighbors(adata)
    sq.gr.spatial_autocorr(
        adata,
        mode="moran",
        genes=selected_genes,
        n_perms=100,
        n_jobs=1,
    )
    sq.gr.spatial_autocorr(
        adata,
        mode="geary",
        genes=selected_genes,
        n_perms=100,
        n_jobs=1,
    )
    moranI = np.median(adata.uns["moranI"]['I'])
    gearyC = np.median(adata.uns["gearyC"]['C'])
    return moranI, gearyC



In [ ]:
import scanpy as sc
import SpaGCN as spg
import os

import warnings
warnings.filterwarnings("ignore")

name = "00_Human_lymph_node_slice1"
adata = sc.read("./results/{}/adata_output.h5ad".format(name))
results_path = "./results_of_post/{}".format(name)
print(adata)

sample_id=adata.obs.index.tolist() 
x_array = adata.obsm["spatial"][:, 0]  
y_array = adata.obsm["spatial"][:, 1]  
adj_2d=spg.calculate_adj_matrix(x=x_array,y=y_array, histology=False)

for i in range(10):
    print("--------------Post the {} results------------------".format(i))
    pred=adata.obs["MISO_labels_{}".format(i)].tolist()
    refined_pred=spg.refine(sample_id=sample_id, pred=pred, dis=adj_2d, shape="hexagon")
    adata.obs["refined_pred_{}".format(i)]=refined_pred
    adata.obs["refined_pred_{}".format(i)]=adata.obs["refined_pred_{}".format(i)].astype('category')


os.makedirs(os.path.join(results_path), exist_ok=True)

print(adata)
adata.write(os.path.join(results_path, 'adata_post_output.h5ad'))

df_results = eval(adata, gt_key="final_annot", spatial_key='spatial', output_csv=os.path.join(results_path, 'eval_results.csv'))